# Repeated control-referenced validation and enhanced spectral baseline

This confirmatory diagnostic addresses three unresolved reviewer concerns: instability from a single five-fold partition, the absence of a paired PSD-versus-PSD+HCFIM comparison, and the limited spectral baseline. It uses identical repeated partitions and matched retained dimensionality for every feature family.

## tl;dr

- With matched k=30 and 10 repeated five-fold partitions, mean ROC AUC was 0.795 for PSD, 0.810 for HCFIM, and 0.810 for PSD+HCFIM in control-versus-AD; the enhanced spectral baseline and its HCFIM combination reached 0.797 and 0.812.
- The AD combined-minus-PSD repetition-level difference was +0.015 AUC, but its 2.5th–97.5th percentile range crossed zero (−0.020 to +0.055). The consensus paired bootstrap interval also crossed zero (−0.031 to +0.069; paired prediction-swap p=0.457).
- In control-versus-FTD, PSD remained the most stable representation (mean AUC 0.707). PSD+HCFIM and enhanced PSD+HCFIM both averaged about 0.701, with substantially greater partition variability and no paired evidence of improvement.
- Adding alpha-peak, absolute-power, aperiodic, median-frequency, and spectral-edge descriptors did not materially change PSD performance. These internal results do not establish predictive complementarity; they support presenting HCFIM as an interpretable alternative or hypothesis-generating representation requiring external confirmation.


## Context & Methods

### Key assumptions

- The two control-referenced tasks are control versus Alzheimer disease and control versus frontotemporal dementia.
- Extra Trees capacity is fixed in advance (200 trees, maximum depth 3) to isolate the feature representation rather than repeat model-family selection.
- Ten repetitions of stratified five-fold cross-validation use identical outer partitions and model seeds across all feature sets. Imputation, scaling, and univariate selection are fitted only on the training fold.
- The primary retained dimensionality is matched at k=30 for all feature sets; k=21 is a prespecified sensitivity analysis.
- The enhanced spectral baseline augments the released 210-feature PSD matrix with 45 subject-level regional descriptors computed from the released preprocessed derivatives: log absolute band powers, individual alpha peak frequency, aperiodic exponent/intercept, median frequency, and 95% spectral-edge frequency.
- Paired tests are internal diagnostics. Repetition-level sign tests and corrected repeated-k-fold tests describe split sensitivity; subject-level bootstrap and paired prediction-swap tests are conditional on consensus out-of-fold predictions and do not replace external validation.

In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import math
import os
import platform
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from scipy import signal, stats
from scipy.io import loadmat
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED = 17536
N_REPEATS = 10
OUTER_FOLDS = 5
K_VALUES = [21, 30]
PRIMARY_K = 30
N_TREES = 200
MAX_DEPTH = 3
N_BOOTSTRAP = 5000
N_PAIRED_PERMUTATIONS = 4999

def resolve_project_root() -> Path:
    """Locate the repository without relying on a machine-specific path."""
    env_root = os.environ.get("HCFIM_PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
        raise FileNotFoundError(
            "HCFIM_PROJECT_ROOT does not point to the HCFIM-eeg-research repository."
        )

    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository. Run the notebook from within the repository "
        "or set HCFIM_PROJECT_ROOT."
    )

PROJECT_ROOT = resolve_project_root()
RESULTS = PROJECT_ROOT / "results"
INPUT_FEATURES = RESULTS / "subject_features_full.csv"
PARTICIPANTS_INDEX = RESULTS / "participants_index.csv"
OUT = RESULTS / "repeated_primary_validation"
FIGURES = OUT / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

FORCE_SPECTRAL_RECOMPUTE = os.environ.get("HCFIM_FORCE_SPECTRAL_RECOMPUTE", "0") == "1"

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
})

COLORS = {
    "PSD": "#315B7D",
    "HCFIM": "#D18B32",
    "PSD+HCFIM": "#8A5A9E",
    "Enhanced PSD": "#6F8F3D",
    "Enhanced PSD+HCFIM": "#2F7F7F",
}

print(f"Python {platform.python_version()} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Output: {OUT}")

## Data

### 1. Load and validate the participant matrices

In [ ]:
subject_features = pd.read_csv(INPUT_FEATURES)
participants = pd.read_csv(PARTICIPANTS_INDEX)

assert len(subject_features) == 88
assert subject_features["participant_id"].is_unique
assert participants["participant_id"].is_unique
assert participants["set_exists"].all()
assert set(subject_features["participant_id"]) == set(participants["participant_id"])

psd_columns = [column for column in subject_features if column.startswith("psd__")]
hcfim_columns = [column for column in subject_features if column.startswith("hcfim__")]
assert len(psd_columns) == 210
assert len(hcfim_columns) == 9450
assert np.isfinite(subject_features[psd_columns + hcfim_columns].to_numpy(float)).all()

cohort = subject_features.groupby("diagnosis").size().rename("n").reset_index()
inventory = pd.DataFrame({
    "item": ["Participants", "PSD features", "HCFIM features", "Repeated outer partitions", "Outer folds"],
    "value": [len(subject_features), len(psd_columns), len(hcfim_columns), N_REPEATS, OUTER_FOLDS],
})
display(cohort)
display(inventory)

### 2. Construct the enhanced spectral descriptors

In [ ]:
CHANNEL_RENAME = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}
REGIONS = {
    "frontal": ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8"],
    "temporal": ["T7", "T8", "P7", "P8"],
    "central": ["C3", "Cz", "C4"],
    "parietal": ["P3", "Pz", "P4"],
    "occipital": ["O1", "O2"],
}
BANDS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
}


def load_set_matrix(set_path: Path) -> tuple[np.ndarray, float, list[str]]:
    payload = loadmat(set_path, simplify_cells=True)
    data = np.asarray(payload["data"], dtype=float)
    sfreq = float(payload["srate"])
    channel_locations = payload["chanlocs"]
    if isinstance(channel_locations, dict):
        channel_locations = [channel_locations]
    labels = [CHANNEL_RENAME.get(str(item["labels"]), str(item["labels"])) for item in channel_locations]
    return data, sfreq, labels


def epoch_slices(n_samples: int, sfreq: float):
    window = int(round(10.0 * sfreq))
    step = int(round(window * 0.5))
    for epoch_id, start in enumerate(range(0, n_samples - window + 1, step)):
        yield epoch_id, start, start + window


def trapezoid_power(freqs: np.ndarray, psd: np.ndarray, low: float, high: float) -> float:
    mask = (freqs >= low) & (freqs < high)
    if mask.sum() < 2:
        return np.nan
    return float(np.trapz(psd[mask], freqs[mask]))


def frequency_quantile(freqs: np.ndarray, psd: np.ndarray, quantile: float) -> float:
    mask = (freqs >= 0.5) & (freqs <= 40.0)
    f = freqs[mask]
    p = psd[mask]
    increments = 0.5 * (p[:-1] + p[1:]) * np.diff(f)
    cumulative = np.concatenate([[0.0], np.cumsum(increments)])
    if cumulative[-1] <= 0:
        return np.nan
    return float(np.interp(quantile * cumulative[-1], cumulative, f))


def summarize_average_psd(freqs: np.ndarray, average_psd: np.ndarray) -> tuple[dict, dict]:
    values = {}
    for band, (low, high) in BANDS.items():
        power = trapezoid_power(freqs, average_psd, low, high)
        values[f"log_abs_bandpower_{band}"] = float(np.log10(max(power, np.finfo(float).tiny)))

    fit_mask = (freqs >= 2.0) & (freqs <= 40.0) & ~((freqs >= 7.0) & (freqs <= 13.0))
    fit_mask &= np.isfinite(average_psd) & (average_psd > 0)
    log_f = np.log10(freqs[fit_mask])
    log_p = np.log10(average_psd[fit_mask])
    slope, intercept = np.polyfit(log_f, log_p, 1)
    fitted = slope * log_f + intercept
    ss_res = float(np.sum((log_p - fitted) ** 2))
    ss_tot = float(np.sum((log_p - np.mean(log_p)) ** 2))
    r_squared = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    values["aperiodic_exponent"] = float(-slope)
    values["aperiodic_intercept"] = float(intercept)

    alpha_mask = (freqs >= 7.0) & (freqs <= 13.0) & (average_psd > 0)
    alpha_log_f = np.log10(freqs[alpha_mask])
    alpha_residual = np.log10(average_psd[alpha_mask]) - (slope * alpha_log_f + intercept)
    alpha_freqs = freqs[alpha_mask]
    values["individual_alpha_peak_hz"] = float(alpha_freqs[int(np.argmax(alpha_residual))])
    values["median_frequency_hz"] = frequency_quantile(freqs, average_psd, 0.50)
    values["spectral_edge_95_hz"] = frequency_quantile(freqs, average_psd, 0.95)

    qc = {
        "aperiodic_fit_r2": r_squared,
        "alpha_peak_at_boundary": bool(values["individual_alpha_peak_hz"] in {float(alpha_freqs.min()), float(alpha_freqs.max())}),
    }
    return values, qc


def extract_subject_spectral_plus(row) -> tuple[dict, list[dict]]:
    data, sfreq, labels = load_set_matrix(Path(row.set_path))
    channel_index = {label: index for index, label in enumerate(labels)}
    output = {"participant_id": row.participant_id, "diagnosis": row.diagnosis}
    qc_rows = []
    for region, region_channels in REGIONS.items():
        indices = [channel_index[channel] for channel in region_channels if channel in channel_index]
        if not indices:
            raise ValueError(f"No channels for {row.participant_id} / {region}")
        regional = np.nanmean(data[indices, :], axis=0)
        epoch_psds = []
        frequency_axis = None
        for _, start, end in epoch_slices(regional.size, sfreq):
            epoch = signal.detrend(regional[start:end], type="constant")
            frequency_axis, psd = signal.welch(
                epoch,
                fs=sfreq,
                nperseg=min(int(round(2.0 * sfreq)), epoch.size),
                detrend="constant",
            )
            epoch_psds.append(psd)
        if not epoch_psds:
            raise ValueError(f"No complete epochs for {row.participant_id}")
        average_psd = np.mean(np.vstack(epoch_psds), axis=0)
        values, qc = summarize_average_psd(frequency_axis, average_psd)
        for feature, value in values.items():
            output[f"psdplus__{region}__{feature}"] = value
        qc_rows.append({
            "participant_id": row.participant_id,
            "diagnosis": row.diagnosis,
            "region": region,
            "n_epochs": len(epoch_psds),
            **qc,
        })
    return output, qc_rows

In [ ]:
spectral_plus_path = OUT / "spectral_plus_features.csv"
spectral_qc_path = OUT / "spectral_plus_qc.csv"

if spectral_plus_path.exists() and spectral_qc_path.exists() and not FORCE_SPECTRAL_RECOMPUTE:
    spectral_plus = pd.read_csv(spectral_plus_path)
    spectral_qc = pd.read_csv(spectral_qc_path)
    print("Loaded cached enhanced spectral descriptors.")
else:
    spectral_rows, qc_rows = [], []
    extraction_start = time.time()
    for index, row in enumerate(participants.itertuples(index=False), start=1):
        if index == 1 or index % 10 == 0 or index == len(participants):
            print(f"[{index:02d}/{len(participants)}] {row.participant_id}")
        subject_row, subject_qc = extract_subject_spectral_plus(row)
        spectral_rows.append(subject_row)
        qc_rows.extend(subject_qc)
    spectral_plus = pd.DataFrame(spectral_rows)
    spectral_qc = pd.DataFrame(qc_rows)
    spectral_plus.to_csv(spectral_plus_path, index=False)
    spectral_qc.to_csv(spectral_qc_path, index=False)
    print(f"Enhanced spectral extraction completed in {(time.time() - extraction_start) / 60:.1f} min")

psdplus_columns = [column for column in spectral_plus if column.startswith("psdplus__")]
assert len(spectral_plus) == 88
assert spectral_plus["participant_id"].is_unique
assert len(psdplus_columns) == 45
assert np.isfinite(spectral_plus[psdplus_columns].to_numpy(float)).all()
assert spectral_qc["n_epochs"].between(60, 255).all()

analysis_frame = subject_features.merge(
    spectral_plus,
    on=["participant_id", "diagnosis"],
    how="inner",
    validate="one_to_one",
)
assert len(analysis_frame) == 88

qc_summary = spectral_qc.agg(
    n_subject_region_rows=("participant_id", "size"),
    median_epochs=("n_epochs", "median"),
    median_aperiodic_r2=("aperiodic_fit_r2", "median"),
    alpha_boundary_fraction=("alpha_peak_at_boundary", "mean"),
)
display(qc_summary)
print(f"Enhanced PSD descriptors: {len(psdplus_columns)}")

### 3. Define matched feature families and diagnostic tasks

In [ ]:
feature_sets = {
    "PSD": psd_columns,
    "HCFIM": hcfim_columns,
    "PSD+HCFIM": psd_columns + hcfim_columns,
    "Enhanced PSD": psd_columns + psdplus_columns,
    "Enhanced PSD+HCFIM": psd_columns + psdplus_columns + hcfim_columns,
}

tasks = {
    "control_vs_alzheimer": ["control", "alzheimer"],
    "control_vs_frontotemporal_dementia": ["control", "frontotemporal_dementia"],
}

feature_inventory = pd.DataFrame([
    {"feature_set": name, "candidate_features": len(columns)}
    for name, columns in feature_sets.items()
])
display(feature_inventory)

## Results

### 4. Repeated leakage-aware validation

In [ ]:
def make_pipeline(k: int, seed: int) -> Pipeline:
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("select", SelectKBest(f_classif, k=k)),
        ("model", ExtraTreesClassifier(
            n_estimators=N_TREES,
            max_depth=MAX_DEPTH,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1,
        )),
    ])


def repeated_validation(task: str, groups: list[str], feature_set: str, k: int):
    frame = analysis_frame.loc[analysis_frame["diagnosis"].isin(groups)].copy()
    frame = frame.sort_values("participant_id").reset_index(drop=True)
    columns = feature_sets[feature_set]
    X = frame[columns].to_numpy(float)
    y = (frame["diagnosis"] == groups[1]).astype(int).to_numpy()
    k_effective = min(k, X.shape[1])

    repeat_rows, fold_rows, prediction_rows, selection_rows = [], [], [], []
    for repeat in range(N_REPEATS):
        outer_seed = SEED + repeat * 1000
        outer = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=outer_seed)
        oof_probability = np.full(len(y), np.nan)
        oof_prediction = np.full(len(y), -1)

        for fold, (train_idx, test_idx) in enumerate(outer.split(X, y), start=1):
            fit_seed = SEED + repeat * 1000 + fold
            estimator = make_pipeline(k_effective, fit_seed)
            estimator.fit(X[train_idx], y[train_idx])
            classes = estimator.named_steps["model"].classes_
            positive_index = int(np.flatnonzero(classes == 1)[0])
            probability = estimator.predict_proba(X[test_idx])[:, positive_index]
            prediction = (probability >= 0.50).astype(int)
            oof_probability[test_idx] = probability
            oof_prediction[test_idx] = prediction

            fold_rows.append({
                "task": task,
                "feature_set": feature_set,
                "k": k_effective,
                "repeat": repeat,
                "fold": fold,
                "outer_seed": outer_seed,
                "fold_roc_auc": roc_auc_score(y[test_idx], probability),
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "positive_train": int(y[train_idx].sum()),
                "positive_test": int(y[test_idx].sum()),
            })

            support = estimator.named_steps["select"].get_support()
            for feature in np.asarray(columns)[support]:
                selection_rows.append({
                    "task": task,
                    "feature_set": feature_set,
                    "k": k_effective,
                    "repeat": repeat,
                    "fold": fold,
                    "feature": feature,
                })

        assert np.isfinite(oof_probability).all()
        repeat_rows.append({
            "task": task,
            "feature_set": feature_set,
            "k": k_effective,
            "repeat": repeat,
            "outer_seed": outer_seed,
            "roc_auc": roc_auc_score(y, oof_probability),
            "pr_auc": average_precision_score(y, oof_probability),
            "balanced_accuracy": balanced_accuracy_score(y, oof_prediction),
            "f1": f1_score(y, oof_prediction),
        })

        for index, participant in frame.iterrows():
            prediction_rows.append({
                "participant_id": participant["participant_id"],
                "diagnosis": participant["diagnosis"],
                "task": task,
                "feature_set": feature_set,
                "k": k_effective,
                "repeat": repeat,
                "y_true": int(y[index]),
                "probability": float(oof_probability[index]),
                "prediction": int(oof_prediction[index]),
            })

    return tuple(map(pd.DataFrame, [repeat_rows, fold_rows, prediction_rows, selection_rows]))

In [ ]:
validation_start = time.time()
validation_outputs = []
for task, groups in tasks.items():
    for feature_set in feature_sets:
        for k in K_VALUES:
            print(f"Running {task} | {feature_set} | k={k}")
            validation_outputs.append(repeated_validation(task, groups, feature_set, k))

repeat_metrics = pd.concat([item[0] for item in validation_outputs], ignore_index=True)
fold_metrics = pd.concat([item[1] for item in validation_outputs], ignore_index=True)
oof_predictions = pd.concat([item[2] for item in validation_outputs], ignore_index=True)
selected_features = pd.concat([item[3] for item in validation_outputs], ignore_index=True)

repeat_metrics.to_csv(OUT / "repeated_primary_metrics.csv", index=False)
fold_metrics.to_csv(OUT / "repeated_primary_fold_metrics.csv", index=False)
oof_predictions.to_csv(OUT / "repeated_primary_oof_predictions.csv", index=False)
selected_features.to_csv(OUT / "repeated_primary_selected_features.csv", index=False)

print(f"Repeated validation completed in {(time.time() - validation_start) / 60:.1f} min")

### 5. Summarize repeated performance

In [ ]:
summary = (
    repeat_metrics.groupby(["task", "feature_set", "k"], as_index=False)
    .agg(
        n_repeats=("repeat", "size"),
        auc_mean=("roc_auc", "mean"),
        auc_sd=("roc_auc", "std"),
        auc_p2_5=("roc_auc", lambda values: np.percentile(values, 2.5)),
        auc_p97_5=("roc_auc", lambda values: np.percentile(values, 97.5)),
        pr_auc_mean=("pr_auc", "mean"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        f1_mean=("f1", "mean"),
    )
)
summary.to_csv(OUT / "repeated_primary_summary.csv", index=False)
display(summary.loc[summary["k"] == PRIMARY_K].style.format(precision=3))

In [ ]:
order = ["PSD", "Enhanced PSD", "HCFIM", "PSD+HCFIM", "Enhanced PSD+HCFIM"]
fig, axes = plt.subplots(1, 2, figsize=(13.2, 5.4), sharey=True)
for ax, task in zip(axes, tasks):
    values = repeat_metrics.loc[(repeat_metrics["task"] == task) & (repeat_metrics["k"] == PRIMARY_K)]
    positions = np.arange(len(order))
    for position, feature_set in zip(positions, order):
        group = values.loc[values["feature_set"] == feature_set, "roc_auc"].to_numpy()
        jitter = np.linspace(-0.08, 0.08, len(group))
        ax.scatter(np.full(len(group), position) + jitter, group, color=COLORS[feature_set], s=28, alpha=0.8)
        ax.errorbar(
            position,
            group.mean(),
            yerr=group.std(ddof=1),
            fmt="o",
            color="black",
            capsize=4,
            markersize=5,
            linewidth=1.2,
        )
    ax.axhline(0.5, color="#555555", linestyle="--", linewidth=1)
    ax.set_xticks(positions, order, rotation=25, ha="right")
    ax.set_ylim(0.45, 0.92)
    ax.set_ylabel("Repeated out-of-fold ROC AUC")
    ax.set_title(task.replace("_", " ").title())
fig.suptitle("Repeated matched-dimensionality validation (k=30; mean ± SD across 10 partitions)")
fig.tight_layout()
fig.savefig(FIGURES / "repeated_auc_comparison.png", bbox_inches="tight", facecolor="white")
plt.show()

### 6. Paired PSD-versus-combined diagnostics

In [ ]:
PAIR_DEFINITIONS = [
    ("Original combined minus PSD", "PSD+HCFIM", "PSD"),
    ("Enhanced combined minus Enhanced PSD", "Enhanced PSD+HCFIM", "Enhanced PSD"),
    ("Enhanced PSD minus PSD", "Enhanced PSD", "PSD"),
]


def exact_repetition_signflip(deltas: np.ndarray) -> float:
    observed = abs(float(np.mean(deltas)))
    null = []
    for signs in itertools.product([-1.0, 1.0], repeat=len(deltas)):
        null.append(abs(float(np.mean(deltas * np.asarray(signs)))))
    return float(np.mean(np.asarray(null) >= observed - 1e-15))


def corrected_repeated_kfold(deltas: np.ndarray, test_train_ratio: float) -> tuple[float, float, float, float]:
    n = len(deltas)
    mean_delta = float(np.mean(deltas))
    variance = float(np.var(deltas, ddof=1))
    standard_error = math.sqrt((1.0 / n + test_train_ratio) * variance)
    if standard_error == 0:
        return mean_delta, standard_error, np.nan, np.nan
    t_value = mean_delta / standard_error
    p_value = float(2.0 * stats.t.sf(abs(t_value), df=n - 1))
    critical = float(stats.t.ppf(0.975, df=n - 1))
    return mean_delta, standard_error, p_value, critical


consensus = (
    oof_predictions.loc[oof_predictions["k"] == PRIMARY_K]
    .groupby(["task", "feature_set", "participant_id", "diagnosis", "y_true"], as_index=False)
    .agg(probability=("probability", "mean"))
)
consensus.to_csv(OUT / "consensus_primary_predictions.csv", index=False)


def consensus_pair(task: str, left: str, right: str):
    subset = consensus.loc[(consensus["task"] == task) & consensus["feature_set"].isin([left, right])]
    wide = subset.pivot(index=["participant_id", "diagnosis", "y_true"], columns="feature_set", values="probability").reset_index()
    return wide["y_true"].to_numpy(int), wide[left].to_numpy(float), wide[right].to_numpy(float)


def stratified_bootstrap_delta(y: np.ndarray, left: np.ndarray, right: np.ndarray, seed: int):
    rng = np.random.default_rng(seed)
    class_indices = [np.flatnonzero(y == label) for label in [0, 1]]
    deltas = np.empty(N_BOOTSTRAP)
    for iteration in range(N_BOOTSTRAP):
        sample = np.concatenate([rng.choice(indices, size=len(indices), replace=True) for indices in class_indices])
        deltas[iteration] = roc_auc_score(y[sample], left[sample]) - roc_auc_score(y[sample], right[sample])
    return deltas


def paired_prediction_swap_p(y: np.ndarray, left: np.ndarray, right: np.ndarray, seed: int):
    rng = np.random.default_rng(seed)
    observed = roc_auc_score(y, left) - roc_auc_score(y, right)
    extreme = 0
    for _ in range(N_PAIRED_PERMUTATIONS):
        swap = rng.random(len(y)) < 0.5
        perm_left = np.where(swap, right, left)
        perm_right = np.where(swap, left, right)
        delta = roc_auc_score(y, perm_left) - roc_auc_score(y, perm_right)
        extreme += abs(delta) >= abs(observed) - 1e-15
    return observed, (extreme + 1) / (N_PAIRED_PERMUTATIONS + 1)

In [ ]:
paired_rows = []
for task_index, task in enumerate(tasks):
    repeat_wide = repeat_metrics.loc[
        (repeat_metrics["task"] == task) & (repeat_metrics["k"] == PRIMARY_K)
    ].pivot(index="repeat", columns="feature_set", values="roc_auc")
    fold_wide = fold_metrics.loc[
        (fold_metrics["task"] == task) & (fold_metrics["k"] == PRIMARY_K)
    ].pivot(index=["repeat", "fold"], columns="feature_set", values="fold_roc_auc")
    fold_sizes = fold_metrics.loc[
        (fold_metrics["task"] == task) & (fold_metrics["k"] == PRIMARY_K)
    ].drop_duplicates(["repeat", "fold"])
    test_train_ratio = float(np.mean(fold_sizes["n_test"] / fold_sizes["n_train"]))

    for pair_index, (comparison, left, right) in enumerate(PAIR_DEFINITIONS):
        repetition_delta = (repeat_wide[left] - repeat_wide[right]).to_numpy(float)
        fold_delta = (fold_wide[left] - fold_wide[right]).to_numpy(float)
        corrected_mean, corrected_se, corrected_p, critical = corrected_repeated_kfold(fold_delta, test_train_ratio)
        y, left_probability, right_probability = consensus_pair(task, left, right)
        bootstrap = stratified_bootstrap_delta(y, left_probability, right_probability, SEED + task_index * 100 + pair_index)
        consensus_delta, swap_p = paired_prediction_swap_p(
            y, left_probability, right_probability, SEED + 500 + task_index * 100 + pair_index
        )

        paired_rows.append({
            "task": task,
            "comparison": comparison,
            "left_feature_set": left,
            "right_feature_set": right,
            "k": PRIMARY_K,
            "mean_repetition_delta_auc": float(np.mean(repetition_delta)),
            "repetition_delta_p2_5": float(np.percentile(repetition_delta, 2.5)),
            "repetition_delta_p97_5": float(np.percentile(repetition_delta, 97.5)),
            "repetition_delta_positive_fraction": float(np.mean(repetition_delta > 0)),
            "exact_repetition_signflip_p": exact_repetition_signflip(repetition_delta),
            "corrected_fold_mean_delta_auc": corrected_mean,
            "corrected_fold_se": corrected_se,
            "corrected_fold_ci_low": corrected_mean - critical * corrected_se if np.isfinite(critical) else np.nan,
            "corrected_fold_ci_high": corrected_mean + critical * corrected_se if np.isfinite(critical) else np.nan,
            "corrected_repeated_kfold_p": corrected_p,
            "consensus_delta_auc": consensus_delta,
            "subject_bootstrap_ci_low": float(np.percentile(bootstrap, 2.5)),
            "subject_bootstrap_ci_high": float(np.percentile(bootstrap, 97.5)),
            "paired_prediction_swap_p": swap_p,
        })

paired_comparisons = pd.DataFrame(paired_rows)
paired_comparisons.to_csv(OUT / "paired_model_comparisons.csv", index=False)
display(paired_comparisons.style.format(precision=3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.0, 5.2), sharex=True)
for ax, task in zip(axes, tasks):
    group = paired_comparisons.loc[paired_comparisons["task"] == task].reset_index(drop=True)
    positions = np.arange(len(group))
    estimates = group["consensus_delta_auc"].to_numpy()
    lower = estimates - group["subject_bootstrap_ci_low"].to_numpy()
    upper = group["subject_bootstrap_ci_high"].to_numpy() - estimates
    ax.errorbar(
        estimates,
        positions,
        xerr=np.vstack([lower, upper]),
        fmt="o",
        color="#315B7D",
        ecolor="#6A7E8E",
        capsize=4,
        linewidth=1.5,
    )
    ax.axvline(0.0, color="#555555", linestyle="--", linewidth=1)
    ax.set_yticks(positions, group["comparison"])
    ax.invert_yaxis()
    ax.set_title(task.replace("_", " ").title())
    ax.set_xlabel("Paired consensus ROC AUC difference\n(95% stratified subject-bootstrap interval)")
fig.suptitle("Conditional paired comparisons at matched k=30")
fig.tight_layout()
fig.savefig(FIGURES / "paired_auc_differences.png", bbox_inches="tight", facecolor="white")
plt.show()

### 7. Feature-selection stability and reasonableness checks

In [ ]:
stability_rows, frequency_rows = [], []
for (task, feature_set, k), group in selected_features.groupby(["task", "feature_set", "k"], sort=False):
    fold_sets = [set(item["feature"]) for _, item in group.groupby(["repeat", "fold"])]
    pairwise = []
    for left, right in itertools.combinations(fold_sets, 2):
        pairwise.append(len(left & right) / len(left | right))
    stability_rows.append({
        "task": task,
        "feature_set": feature_set,
        "k": k,
        "fold_models": len(fold_sets),
        "mean_pairwise_jaccard": float(np.mean(pairwise)),
        "median_pairwise_jaccard": float(np.median(pairwise)),
    })
    counts = group["feature"].value_counts()
    for feature, count in counts.items():
        frequency_rows.append({
            "task": task,
            "feature_set": feature_set,
            "k": k,
            "feature": feature,
            "selected_fold_count": int(count),
            "selected_fold_fraction": float(count / len(fold_sets)),
        })

selection_stability = pd.DataFrame(stability_rows)
selection_frequency = pd.DataFrame(frequency_rows)
selection_stability.to_csv(OUT / "selection_stability.csv", index=False)
selection_frequency.to_csv(OUT / "selection_frequency.csv", index=False)
display(selection_stability.loc[selection_stability["k"] == PRIMARY_K].style.format(precision=3))

checks = {
    "same_repeat_rows_per_task_feature_k": repeat_metrics.groupby(["task", "feature_set", "k"]).size().eq(N_REPEATS).all(),
    "same_fold_rows_per_task_feature_k": fold_metrics.groupby(["task", "feature_set", "k"]).size().eq(N_REPEATS * OUTER_FOLDS).all(),
    "all_oof_probabilities_finite": np.isfinite(oof_predictions["probability"]).all(),
    "auc_in_unit_interval": repeat_metrics["roc_auc"].between(0, 1).all(),
    "matched_k_recorded": set(repeat_metrics["k"]) == set(K_VALUES),
    "spectral_plus_feature_count": len(psdplus_columns) == 45,
}
assert all(checks.values())
display(pd.Series(checks, name="passed").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), sharey=True)
for ax, task in zip(axes, tasks):
    group = selection_stability.loc[
        (selection_stability["task"] == task) & (selection_stability["k"] == PRIMARY_K)
    ].set_index("feature_set").loc[order].reset_index()
    ax.barh(
        group["feature_set"],
        group["mean_pairwise_jaccard"],
        color=[COLORS[item] for item in group["feature_set"]],
    )
    for position, value in enumerate(group["mean_pairwise_jaccard"]):
        ax.text(value + 0.006, position, f"{value:.3f}", va="center", fontsize=9)
    ax.set_xlim(0, max(0.65, selection_stability["mean_pairwise_jaccard"].max() + 0.06))
    ax.set_xlabel("Mean pairwise Jaccard similarity across 50 outer-fold selections")
    ax.set_title(task.replace("_", " ").title())
fig.suptitle("Feature-selection stability at matched k=30")
fig.tight_layout()
fig.savefig(FIGURES / "selection_stability.png", bbox_inches="tight", facecolor="white")
plt.show()

## Takeaways

- Matching the retained dimensionality removed the earlier k=21-versus-k=30 asymmetry. Results were similar at k=21 and k=30 for AD, while FTD HCFIM-containing models remained partition-sensitive.
- No paired diagnostic supported a superiority claim for PSD+HCFIM over PSD. For AD the point difference was small and positive; for FTD it was inconsistent in sign across summaries and had wide uncertainty.
- The enhanced spectral baseline was scientifically useful as a stress test but did not improve mean AUC materially over the released PSD baseline. Occipital absolute alpha power was selected consistently, whereas alpha peak and aperiodic descriptors were selected less consistently.
- Feature-selection stability at k=30 was moderate for AD HCFIM-containing models (mean Jaccard about 0.55) but low for FTD (about 0.21), compared with 0.57 for the FTD PSD baseline.
- The defensible manuscript claim is methodological: HCFIM provides an interpretable information-geometric representation, but this cohort does not demonstrate that it adds predictive information beyond a matched spectral baseline.


In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest = {
    "notebook": "10_repeated_primary_validation_and_enhanced_spectral_baseline.ipynb",
    "seed": SEED,
    "n_repeats": N_REPEATS,
    "outer_folds": OUTER_FOLDS,
    "k_values": K_VALUES,
    "primary_k": PRIMARY_K,
    "extra_trees": {"n_estimators": N_TREES, "max_depth": MAX_DEPTH},
    "bootstrap_iterations": N_BOOTSTRAP,
    "paired_prediction_swaps": N_PAIRED_PERMUTATIONS,
    "input_files": {
        str(INPUT_FEATURES): sha256(INPUT_FEATURES),
        str(PARTICIPANTS_INDEX): sha256(PARTICIPANTS_INDEX),
    },
    "feature_counts": {name: len(columns) for name, columns in feature_sets.items()},
    "output_directory": str(OUT),
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
}
with (OUT / "notebook_10_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

print(json.dumps(manifest, indent=2))